# RAG for Predictive Maintenance (LIME Incoporation) - Local Interpretable Model-agnostic Explanations

### Table of Contents (Pipeline)

| Section | Purpose |
|---------|----------|
| **Part 0 — Setup** | Environment prep, folder structure, dependency validation |
| **Part 1 — PdM Model + Saftey Threshold + LIME** | Train interpretable PdM baseline model + local explainability |
| **Part 2 - Train/Test spilt + scale once + SMOTE + RF + calibration** | Train/Test Split of Working Data |
| **Part 3 — Bayesian Confidence Scoring** | For Model Confidence Scoring calibration | 
| **Part 4 — Safety Threshold Computation** | Incoporating Saftey Threshold as safety measure  | 
| **Part 5 — Build LIME Explainer** | To be consistent with scaled features for analysis | 
| **Part 6 — LIME Safety Threshold Rules** | Learning this from failures |
| **Part 7 — LIME-risk score function** | Convert LIME thresholds into a LIME-risk score function |
| **Part 8 — Manual Ingestion** | Engineering manual PDFs → structured chunks + SQLite store | 
| **Part 9 — Embeddings & Index** | Build vector + lexical indexes (FAISS or TF-IDF fallback) | 
| **Part 10 — Hybrid Retrieval Pipeline** | Hybrid retrieval: vector + keyword + BM25 + context packing | 
| **Part 11 — Two Stage LLM Prompts (Ollama)** | Stage1 classification + Stage2 engineering reasoning + citations | 
| **Part 12 — Post-processing** | Validate JSON, filter hallucinations, format operator output | 
| **Part 13 — Audio Output (TTS)** | Convert explanations to offline-playable audio | 

##### It tries to explain the predictions of any complex, "black-box" machine learning model by building simpler, 
##### interpretable models locally around a specific prediction.
##### The core idea is that while a complex model's behavior might be highly non-linear globally, it can be approximated by a  simple, 
##### understandable model (like linear regression or a decision tree) in the immediate vicinity of a specific data point.

##### HOW IT WORKS
##### Choose a specific data point (instance) from the dataset for which you want to explain the black-box model's prediction.
##### slightly modified (perturbed) data samples around the chosen instance. The method for perturbation depends on the data type.
##### Pass all the perturbed samples through the original, complex black-box model to get their predictions. 
##### This creates a new dataset of inputs and corresponding outputs for the local explanation model to learn from.
##### Assign a weight to each new sample based on its proximity to the original instance of interest. 
##### Train an interpretable model on the weighted, perturbed dataset.
##### This local surrogate model approximates the black-box model's behavior in that specific neighborhood. 
##### The coefficients or rules from this simple model are then presented as the explanation.

## 0) Setup — folders, pip installs, capability report

In [356]:
# 0) Setup — folders + imports + reproducibility
import os, sys, json, math, re, random, sqlite3, subprocess, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
RAW = DATA / "raw"
CLEAN = DATA / "clean"
INDEX = DATA / "index"
DB_PATH = INDEX / "rag_pdms.sqlite"

for p in [DATA, RAW, CLEAN, INDEX]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready:", DATA, RAW, CLEAN, INDEX, sep="\n- ")

Folders ready:
- /Users/martinwalterampah/Desktop/Final_Research_Project/data
- /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw
- /Users/martinwalterampah/Desktop/Final_Research_Project/data/clean
- /Users/martinwalterampah/Desktop/Final_Research_Project/data/index


## 1) Install/Imports (only once)

In [163]:
# (Optional) pip installs - run once if needed
# !pip install -q pandas numpy scikit-learn imbalanced-learn lime pypdf rank_bm25 sentence_transformers faiss-cpu tiktoken joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score, average_precision_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

from lime.lime_tabular import LimeTabularExplainer


## PART 1 — PdM MODEL + SAFETY THRESHOLD + LIME “RISK RULE”

In [164]:
CSV_PATH = Path("Data_file/ai4i2020.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")

df_raw = pd.read_csv(CSV_PATH)

# Detect columns
COLS = {"air_temperature":None,"process_temperature":None,"rotational_speed":None,"torque":None,"tool_wear":None,"label":None}
ALIASES = {
    "air_temperature":["air temperature","air_temperature"],
    "process_temperature":["process temperature","process_temperature"],
    "rotational_speed":["rotational speed","rotational_speed","rpm"],
    "torque":["torque","[nm]"],
    "tool_wear":["tool wear","tool_wear","[min]"],
    "label":["machine failure","machine_failure","failure","target","label"]
}

for k, alts in ALIASES.items():
    for c in df_raw.columns:
        if any(a in c.lower() for a in alts):
            COLS[k]=c
            break

if COLS["label"] is None:
    raise ValueError("Could not detect label column. Rename to 'Machine failure' or update ALIASES['label'].")

print("Detected:", json.dumps(COLS, indent=2))

df = df_raw.copy()

# Create engineered features
df["temp_diff"] = df[COLS["process_temperature"]] - df[COLS["air_temperature"]]
df["rot_rads"]  = (df[COLS["rotational_speed"]] * 2 * math.pi) / 60.0
df["power_W"]   = df[COLS["torque"]] * df["rot_rads"]
df["overstrain_metric"] = df[COLS["torque"]] * df[COLS["tool_wear"]]

LABEL = COLS["label"]
y = df[LABEL].astype(int)

# Select numeric features only (drop Product ID / Type)
drop_non_numeric = [c for c in ["Product ID", "Type"] if c in df.columns]
X = df.drop(columns=[LABEL] + drop_non_numeric, errors="ignore")

# Ensure all numeric
X = X.select_dtypes(include=["number"]).copy()

print("X shape:", X.shape, "y mean (failure rate):", y.mean())
X.head()

Detected: {
  "air_temperature": "Air temperature [K]",
  "process_temperature": "Process temperature [K]",
  "rotational_speed": "Rotational speed [rpm]",
  "torque": "Torque [Nm]",
  "tool_wear": "Tool wear [min]",
  "label": "Machine failure"
}
X shape: (10000, 15) y mean (failure rate): 0.0339


,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF,temp_diff,rot_rads,power_W,overstrain_metric
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,10.5,162.420340,6951.590560,0.0
1,2,298.2,308.7,1408,46.3,3,0,0,0,0,0,10.5,147.445415,6826.722724,138.9
2,3,298.1,308.5,1498,49.4,5,0,0,0,0,0,10.4,156.870193,7749.387543,247.0
3,4,298.2,308.6,1433,39.5,7,0,0,0,0,0,10.4,150.063409,5927.504659,276.5
4,5,298.2,308.7,1408,40.0,9,0,0,0,0,0,10.5,147.445415,5897.816608,360.0


## CELL 3 — Train/test split + scale once + SMOTE + RF + calibration

In [165]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Scale numeric features (single scaler used everywhere)
scaler = StandardScaler()
X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
X_te_s = pd.DataFrame(scaler.transform(X_te), columns=X_te.columns, index=X_te.index)

# SMOTE on training only
sm = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_tr_s, y_tr)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
).fit(X_res, y_res)

# calibrated probabilities (important for threshold tuning)
cal_rf = CalibratedClassifierCV(rf, cv=3, method="isotonic").fit(X_res, y_res)

probs_te = cal_rf.predict_proba(X_te_s)[:, 1]

print("RF+Calibrated evaluation @0.5 (baseline only):")
#Receiver Operating Characteristic - Area Under Curve
print(classification_report(y_te, (probs_te >= 0.5).astype(int), digits=3))
print("ROC AUC:", roc_auc_score(y_te, probs_te), "AP:", average_precision_score(y_te, probs_te))

RF+Calibrated evaluation @0.5 (baseline only):
              precision    recall  f1-score   support

           0      0.999     0.999     0.999      1932
           1      0.971     0.971     0.971        68

    accuracy                          0.998      2000
   macro avg      0.985     0.985     0.985      2000
weighted avg      0.998     0.998     0.998      2000

ROC AUC: 0.9903064486664231 AP: 0.9743725490196079


# Bayesian Confidence Scoring 

In [166]:
# ============================================================
# BAYESIAN CONFIDENCE SCORING
# ============================================================

def bayesian_confidence(prob, evidence_strength=20):
    """
    Bayesian confidence estimation using Beta distribution.
    
    prob: predicted probability of failure
    evidence_strength: controls certainty of prediction
    
    returns:
        confidence score in range [0,1]
    """
    
    # Evidence parameters
    alpha = 1 + evidence_strength * prob
    beta = 1 + evidence_strength * (1 - prob)
    
    # Beta distribution variance
    variance = (alpha * beta) / (((alpha + beta)**2) * (alpha + beta + 1))
    
    # Maximum variance for Beta distribution occurs at alpha=beta=1
    max_variance = 1/12
    
    confidence = 1 - (variance / max_variance)
    
    return float(np.clip(confidence, 0, 1))

### Add Confidence Categories
### This makes results easier for engineers to interpret.

In [167]:
def confidence_level(conf):
    
    if conf >= 0.85:
        return "Very High Confidence"
    
    elif conf >= 0.70:
        return "High Confidence"
    
    elif conf >= 0.50:
        return "Moderate Confidence"
    
    else:
        return "Low Confidence"

## CELL 4 — Compute a SAFETY operating threshold (cost-sensitive)

In [168]:
def choose_tau_cost_sensitive(y_true, probs, C_FN=30.0, C_FP=1.0):
    # Search a grid of thresholds
    taus = np.linspace(0.01, 0.99, 99)
    best = None
    for tau in taus:
        yhat = (probs >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()
        cost = C_FN * fn + C_FP * fp
        if best is None or cost < best["cost"]:
            best = {"tau": float(tau), "cost": float(cost), "tn": tn, "fp": fp, "fn": fn, "tp": tp}
    return best

best_tau = choose_tau_cost_sensitive(y_te.values, probs_te, C_FN=30.0, C_FP=1.0)
tau_safety = best_tau["tau"]

print("Chosen SAFETY threshold tau_safety =", tau_safety)
print("At tau_safety:", best_tau)
print(classification_report(y_te, (probs_te >= tau_safety).astype(int), digits=3))

Chosen SAFETY threshold tau_safety = 0.64
At tau_safety: {'tau': 0.64, 'cost': 60.0, 'tn': np.int64(1932), 'fp': np.int64(0), 'fn': np.int64(2), 'tp': np.int64(66)}
              precision    recall  f1-score   support

           0      0.999     1.000     0.999      1932
           1      1.000     0.971     0.985        68

    accuracy                          0.999      2000
   macro avg      0.999     0.985     0.992      2000
weighted avg      0.999     0.999     0.999      2000



## CELL 5 — Build LIME Explainer (consistent with scaled features)

In [169]:
FEATURE_ORDER = list(X_tr_s.columns)
CLASS_NAMES = ["Normal", "Failure"]

lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_res),          # same space as model training
    feature_names=FEATURE_ORDER,
    class_names=CLASS_NAMES,
    discretize_continuous=True,
    mode="classification",
    random_state=RANDOM_STATE
)

def lime_weights_to_percent(exp_list, top_k=5):
    """
    Convert LIME weights into normalised percentage contributions.
    Returns a sorted list of dicts.
    """
    clean = []
    for desc, w in exp_list:
        feat = desc.split()[0]  # extract feature name
        clean.append((feat, float(w)))

    # aggregate duplicate features
    agg = {}
    for feat, w in clean:
        agg[feat] = agg.get(feat, 0.0) + abs(w)

    total = sum(agg.values()) + 1e-9

    # normalize to %
    out = [
        {"feature": f, "weight": w, "percent": (w / total) * 100}
        for f, w in agg.items()
    ]

    # sort descending
    out = sorted(out, key=lambda x: x["percent"], reverse=True)

    return out[:top_k]

def format_lime_explanation(percent_list): #Human Readable Formatter
    """
    Convert percentages into a natural language sentence.
    """
    parts = [
        f"{round(p['percent'])}% by {p['feature'].replace('_',' ').title()}"
        for p in percent_list
    ]
    return "This prediction is influenced " + ", ".join(parts) + "."

def lime_explain_one(x_row_scaled, num_features=8):
    """
    x_row_scaled: 1D array in the same scaled feature space as the model (X_te_s row)
    returns list of (feature_description, weight)
    """
    x_row_scaled = np.asarray(x_row_scaled, dtype=float).reshape(-1)

    def prob_fn(X_np):
        dfX = pd.DataFrame(X_np, columns=FEATURE_ORDER)
        P = cal_rf.predict_proba(dfX)[:, 1]
        return np.column_stack([1.0 - P, P])

    exp = lime_explainer.explain_instance(
        data_row=x_row_scaled,
        predict_fn=prob_fn,
        num_features=num_features
    )
    return exp.as_list()

## CELL 6 — Learn “LIME Safety Threshold Rules” from failures

In [170]:
def build_lime_threshold_table(X_ref_scaled, y_ref, n_samples=250, topk=6, pos_only=True):
    """
    Build a per-feature threshold table from LIME explanations over true failures.
    Returns dict: feature -> {q_hi, q_lo, direction}
    """
    # select true failures
    fail_idx = np.where(y_ref.values == 1)[0]
    if len(fail_idx) == 0:
        raise ValueError("No failures found in reference set for LIME threshold building.")

    # sample failures for efficiency
    take = fail_idx if len(fail_idx) <= n_samples else np.random.choice(fail_idx, n_samples, replace=False)

    rows = []
    for i in take:
        x_row = X_ref_scaled.iloc[i].values
        exp_list = lime_explain_one(x_row, num_features=topk)

        for desc, w in exp_list:
            # desc like: "torque <= 0.31" or "power_W > -0.12"
            feat = desc.split()[0]
            if feat not in FEATURE_ORDER:
                continue
            if pos_only and w <= 0:
                continue
            rows.append({"feature": feat, "weight": w, "value": float(X_ref_scaled.iloc[i][feat])})

    lime_df = pd.DataFrame(rows)
    if lime_df.empty:
        raise ValueError("LIME produced no usable positive contributions. Try pos_only=False or increase n_samples.")

    # For each feature: compute high-risk thresholds based on the distribution of values where it contributes positively.
    table = {}
    for feat in lime_df["feature"].unique():
        vals = lime_df[lime_df["feature"] == feat]["value"].values
        # Because everything is scaled, we work in scaled space here; report later in raw space if needed.
        table[feat] = {
            "q50": float(np.percentile(vals, 50)),
            "q70": float(np.percentile(vals, 70)),
            "q85": float(np.percentile(vals, 85)),
            "n": int(len(vals))
        }
    return table, lime_df

lime_thresholds_scaled, lime_support_df = build_lime_threshold_table(
    X_ref_scaled=X_te_s.reset_index(drop=True),
    y_ref=y_te.reset_index(drop=True),
    n_samples=250, topk=6, pos_only=True
)

print("Example LIME threshold table (scaled-space):")
for k in list(lime_thresholds_scaled.keys())[:8]:
    print(k, lime_thresholds_scaled[k])

Example LIME threshold table (scaled-space):
HDF {'q50': 9.592875263129063, 'q70': 9.592875263129063, 'q85': 9.592875263129063, 'n': 29}
temp_diff {'q50': -1.6009460319384439, 'q70': -1.5009251941692283, 'q85': -1.4009043564000696, 'n': 31}
PWF {'q50': 9.82654443890405, 'q70': 9.82654443890405, 'q85': 9.82654443890405, 'n': 13}
power_W {'q50': 1.9395776790026478, 'q70': 2.2837827916464657, 'q85': 2.6656248559047464, 'n': 32}
overstrain_metric {'q50': 2.4668487194578206, 'q70': 2.694267078716311, 'q85': 2.886820202082744, 'n': 30}
OSF {'q50': 9.826544438904051, 'q70': 9.826544438904051, 'q85': 9.826544438904051, 'n': 16}
TWF {'q50': 14.873541011548731, 'q70': 14.873541011548731, 'q85': 14.873541011548731, 'n': 10}


## CELL 7 — Convert LIME thresholds into a LIME-risk score function

In [171]:
# pick top N features from LIME support frequency
top_feats = (
    lime_support_df["feature"].value_counts().head(8).index.tolist()
)

def lime_risk_score(x_scaled_row, thresholds_table, features=top_feats, quantile_key="q70"):
    risk = 0
    triggers = []
    for feat in features:
        if feat not in thresholds_table:
            continue
        thr = thresholds_table[feat][quantile_key]
        val = float(x_scaled_row[feat])
        direction = "high" if thr > 0 else "low"
        if (direction == "high" and val > thr) or (direction == "low" and val < thr):
            risk += 1
            triggers.append({"feature": feat, "value": val, "thr": thr})
    return risk, triggers

# tune "min_hits" to make it a safety-oriented gate
MIN_HITS = 2

## PART 2 — Manual ingestion

## CELL 8 — Loading of the Engineering Manual pdfs for ingestion

In [172]:
# Bridge: copy your Reference_data PDFs into data/raw/ so Step 2 finds them
import shutil
from pathlib import Path

# Adjust this if your notebook is not inside Final_Research_Project
# If you ARE already running inside Final_Research_Project, just use Path("Reference_data")
REF_DIR_CANDIDATES = [
    Path("Reference_data"),
    Path("../Final_Research_Project/Reference_data"),
    Path("../../Final_Research_Project/Reference_data"),
]

REF_DIR = next((p for p in REF_DIR_CANDIDATES if p.exists()), None)
assert REF_DIR is not None, f"Could not find Reference_data at any of: {REF_DIR_CANDIDATES}"

RAW.mkdir(parents=True, exist_ok=True)

expected = {
    "Basics_of_Rotating_Industrial_Equipment.pdf",
    "Engineers_Guide_to_Rotating_Equipment-The_Pocket_Reference.pdf",
    "Machine_Maintenance_Manual.pdf",
    "Operation_and_Maintenance_Manual.pdf",
    "MAINTENANCE_OF_MECHANICAL_AND_ELECTRICAL_EQUIPMENT.pdf",
}

found = []
for fn in expected:
    src = REF_DIR / fn
    if not src.exists():
        print(f"[warn] Not found in Reference_data: {fn}")
        continue
    dst = RAW / fn
    if not dst.exists():
        shutil.copyfile(src, dst)
        print(f"[ok] Copied -> {dst}")
    else:
        print(f"[ok] Already present -> {dst}")
    found.append(dst)

print(f"\nReady to ingest {len(found)} manuals from {RAW}")
for p in RAW.glob("*.pdf"):
    print(" -", p.name)


[ok] Already present -> /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw/Machine_Maintenance_Manual.pdf
[ok] Already present -> /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw/Operation_and_Maintenance_Manual.pdf
[ok] Already present -> /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw/MAINTENANCE_OF_MECHANICAL_AND_ELECTRICAL_EQUIPMENT.pdf
[ok] Already present -> /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw/Engineers_Guide_to_Rotating_Equipment-The_Pocket_Reference.pdf
[ok] Already present -> /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw/Basics_of_Rotating_Industrial_Equipment.pdf

Ready to ingest 5 manuals from /Users/martinwalterampah/Desktop/Final_Research_Project/data/raw
 - Machine_Maintenance_Manual.pdf
 - Basics_of_Rotating_Industrial_Equipment.pdf
 - MAINTENANCE_OF_MECHANICAL_AND_ELECTRICAL_EQUIPMENT.pdf
 - Engineers_Guide_to_Rotating_Equipment-The_Pocket_Reference.pdf
 - Operation_and_Mainte

## Manual ingestion — PDF -> chunks -> SQLite + clean/

In [173]:

import re, datetime, sqlite3
conn = sqlite3.connect(DB_PATH); cur=conn.cursor()
cur.execute("CREATE TABLE IF NOT EXISTS chunks (id INTEGER PRIMARY KEY, manual_id TEXT, title TEXT, section TEXT, page_start INT, page_end INT, text TEXT, metadata_json TEXT)")
conn.commit()

def approx_token_count(text):
    try:
        import tiktoken; enc=tiktoken.get_encoding("cl100k_base"); return len(enc.encode(text))
    except Exception:
        return int(len(text.split())*1.3)

def split_by_structure(paras,tmin=300,tmax=800,overlap=0.15):
    out=[]; buf=[]; tok=0
    for para in paras:
        if not para.strip(): continue
        t=approx_token_count(para)
        if tok+t>tmax and tok>=tmin:
            out.append("\n".join(buf).strip())
            tail=max(1,int(len(buf)*overlap)); buf=buf[-tail:]; tok=sum(approx_token_count(x) for x in buf)
        buf.append(para); tok+=t
    if buf: out.append("\n".join(buf).strip())
    return out

def ingest_pdf(pdf_path: Path):
    try:
        from pypdf import PdfReader
    except Exception:
        print("[note] pypdf missing; skip:", pdf_path.name); return
    reader=PdfReader(str(pdf_path))
    pages=[]
    for i,p in enumerate(reader.pages):
        try: pages.append((i+1,p.extract_text() or ""))
        except Exception: pages.append((i+1,""))
    paras=[]
    for pno,txt in pages:
        for ln in (txt or "").splitlines():
            l=ln.strip()
            if not l: continue
            if re.match(r"^(chapter|section|introduction)\b|\b(safety|maintenance|pump|compressor|bearing|vibration)\b", l.lower()):
                paras.append("### "+l)
            else: paras.append(l)
    #chunks=split_by_structure(paras,300,800,0.15)
    chunks = split_by_structure(paras, tmin=120, tmax=240, overlap=0.20)
    title=pdf_path.stem; manual_id=title[:40].lower().replace(" ","_"); created=datetime.datetime.utcnow().isoformat()
    (CLEAN/ f"{manual_id}.txt").write_text("\n\n".join(chunks),encoding="utf-8")
    for ch in chunks:
        sec=None
        for ln in ch.splitlines():
            if ln.startswith("### "): sec=ln.strip("# ").strip(); break
        meta={"manual_id":manual_id,"created_at":created,"source_pdf":pdf_path.name}
        cur.execute("INSERT INTO chunks (manual_id,title,section,page_start,page_end,text,metadata_json) VALUES (?,?,?,?,?,?,?)",
                    (manual_id,title,sec,1,1,ch,json.dumps(meta)))
    conn.commit()
    print(f"Ingested {pdf_path.name}: {len(chunks)} chunks")

from pathlib import Path
for pdf in (Path("data/raw").glob("*.pdf")):
    ingest_pdf(pdf)

Ingested Machine_Maintenance_Manual.pdf: 6 chunks
Ingested Basics_of_Rotating_Industrial_Equipment.pdf: 106 chunks
Ingested MAINTENANCE_OF_MECHANICAL_AND_ELECTRICAL_EQUIPMENT.pdf: 487 chunks
Ingested Engineers_Guide_to_Rotating_Equipment-The_Pocket_Reference.pdf: 529 chunks
Ingested Operation_and_Maintenance_Manual.pdf: 39 chunks


## PART 3 — Embeddings & Index (keep your offline local ST)

In [174]:
pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [175]:
pip install huggingface_hub

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [176]:
# --- Visibility & progress utilities (MUST RUN FIRST) ------------------------
import os, sys, time, math
from contextlib import contextmanager

def log(msg):
    print(msg, flush=True)

@contextmanager
def timed(label):
    t0 = time.perf_counter()
    log(f"[start] {label} …")
    try:
        yield
    finally:
        t1 = time.perf_counter()
        log(f"[done ] {label} ({t1 - t0:.1f}s)")

# Optional: psutil RAM helper...
try:
    import psutil
    def mem():
        return f"{psutil.Process(os.getpid()).memory_info().rss/1024/1024:.1f} MB"
except Exception:
    psutil=None
    def mem(): return "n/a"

# -----------------------------------------------------------------------

# --- LOCAL ST MODEL CONFIG (no network) --------------------------------------
from pathlib import Path
MODEL_DIR = Path("/Users/martinwalterampah/huggingfolder/mymodel")  # <-- change this to your actual local folder
# Example: Path("/Users/you/Downloads/all-MiniLM-L6-v2")

# Force offline behavior so nothing tries to reach the internet
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# Minimal sanity check of expected files/folders in a SentenceTransformers repo
REQUIRED_ANY = [
    "config.json",
    "model.safetensors",             # or "pytorch_model.bin" depending on the repo you saved
    "modules.json",
]
REQUIRED_POOLING_DIR = MODEL_DIR / "1_Pooling"  # typical ST layout
MISSING = [f for f in REQUIRED_ANY if not (MODEL_DIR / f).exists()]
if not REQUIRED_POOLING_DIR.exists():
    MISSING.append("1_Pooling/")

if MISSING:
    log(f"[warn ] Local ST model folder seems incomplete: missing {MISSING}")
    log(f"        MODEL_DIR = {MODEL_DIR.resolve()}")
else:
    log(f"[ok   ] Found local ST model files at {MODEL_DIR.resolve()}")

# --- Embedding builder with visible progress (OFFLINE) -----------------------
def build_embeddings(texts):
    """Load SentenceTransformers from a LOCAL directory only; else TF-IDF fallback."""
    try:
        from sentence_transformers import SentenceTransformer
        import torch
        # pick best available device
        if torch.backends.mps.is_available():
            device = "mps"
        elif torch.cuda.is_available():
            device = "cuda"
        else:
            device = "cpu"

        # IMPORTANT: pass the LOCAL PATH, not the model id
        log(f"[info ] Loading local SentenceTransformer('{MODEL_DIR}') on device='{device}' …")
        with timed("Load ST model (local)"):
            st = SentenceTransformer(str(MODEL_DIR), device=device)  # <--- local path

        # Manual batching + tqdm progress (reliable in Jupyter)
        try:
            from tqdm.auto import tqdm
        except Exception:
            def tqdm(x, **kw): return x

        batch_size = 256 if device in ("mps","cuda") else 64
        dim = st.get_sentence_embedding_dimension()
        N = len(texts)
        emb = np.empty((N, dim), dtype=np.float32)

        log(f"[info ] Embedding {N} chunks in batches of {batch_size} (dim={dim}). RAM: {mem()}")
        with timed("Encode (SentenceTransformers, local)"):
            import torch
            with torch.inference_mode():
                idx = 0
                for i in tqdm(range(0, N, batch_size), desc="Embedding", leave=True):
                    batch = texts[i:i+batch_size]
                    batch_emb = st.encode(
                        batch,
                        convert_to_numpy=True,
                        normalize_embeddings=True,
                        show_progress_bar=False,   # tqdm handles the bar
                        batch_size=batch_size
                    )
                    emb[idx:idx+len(batch_emb)] = batch_emb
                    idx += len(batch_emb)

        log(f"[info ] Embedding complete. RAM: {mem()}")
        return {"backend":"st","model":"all-MiniLM-L6-v2","emb":emb,"device":device}

    except Exception as e:
        log(f"[note ] Using TF-IDF fallback (local ST load failed). Reason: {e!r}")
        with timed("TF-IDF fit_transform"):
            vec = TfidfVectorizer(max_features=4096, ngram_range=(1,2))
            mat = vec.fit_transform(texts).astype(np.float32)
        log(f"[info ] TF-IDF matrix shape: {mat.shape}. RAM: {mem()}")
        return {"backend":"tfidf","vec":vec,"emb":mat}


[ok   ] Found local ST model files at /Users/martinwalterampah/huggingfolder/mymodel


=====
PART 4 — Safety-Aware Retrieval
=========================

## CELL 10 — Retrieval: add “safety bias re-ranking”

In [177]:
# ---- Safety keyword sets used to bias retrieval ranking ----
SAFETY_TERMS = [
    "lockout", "tagout", "shutdown", "isolation", "hazard", "danger",
    "safety", "ppe", "permit", "confined space", "emergency stop"
]
FAILURE_TERMS = [
    "overstrain", "overload", "torque", "bearing", "vibration", "misalignment",
    "heat dissipation", "overheat", "temperature", "lubrication", "power", "rpm"
]

def safety_rerank(chunks_df: pd.DataFrame, boost_terms, boost_factor=0.85):
    """
    Multiply distance by boost_factor (smaller is better) if text matches safety terms.
    """
    if chunks_df.empty:
        return chunks_df
    pat = re.compile("|".join(re.escape(t) for t in boost_terms), flags=re.IGNORECASE)
    d = chunks_df.copy()
    hit = d["text"].fillna("").apply(lambda t: bool(pat.search(t)))
    # reduce distance for hits => better rank
    if "hybrid_dist" in d.columns:
        d.loc[hit, "hybrid_dist"] = d.loc[hit, "hybrid_dist"] * boost_factor
    d["safety_hit"] = hit.astype(int)
    return d.sort_values(["hybrid_dist", "safety_hit"], ascending=[True, False])

In [178]:
# ============================================================
# CONTEXT PACKING (Jupyter-safe): pack_context + token counter
# ============================================================

def approx_token_count(text: str) -> int:
    """
    Conservative approximate token count.
    Uses tiktoken if available, else word-based approximation.
    """
    try:
        import tiktoken
        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text))
    except Exception:
        return int(len(text.split()) * 1.3)

def pack_context(chunks_df: pd.DataFrame, token_budget: int = 1200):
    """
    Converts retrieved chunks into one context string + citation metadata list.
    Expects columns: id, title, section, page_start, page_end, text
    Returns:
      - context_block (str)
      - citations_meta (list[dict])
    """
    if chunks_df is None or chunks_df.empty:
        return "", []

    seen = set()
    packed = []
    meta = []
    total = 0

    for _, r in chunks_df.iterrows():
        txt = (r.get("text") or "").strip()
        if not txt or txt in seen:
            continue

        need = approx_token_count(txt)
        if total + need > token_budget:
            break

        seen.add(txt)
        total += need

        title = r.get("title") or "Unknown Manual"
        section = r.get("section") or "General"
        p1 = r.get("page_start") or "?"
        p2 = r.get("page_end") or "?"
        header = f"[{title} :: {section} :: pp.{p1}-{p2} :: chunk_id={int(r.get('id'))}]"

        packed.append(header + "\n" + txt)

        meta.append({
            "id": int(r.get("id")) if pd.notna(r.get("id")) else None,
            "manual": title,
            "section": section,
            "pages": f"{p1}-{p2}"
        })

    return "\n\n---\n\n".join(packed), meta

## NB: LLM + Prompts

In [348]:
# ============================================================
# LLM + PROMPTS (Jupyter-safe): run_ollama + stage1_prompt + stage2_prompt
# ============================================================

import subprocess, json

def ollama_available():
    try:
        r = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=3)
        return (r.returncode == 0)
    except Exception:
        return False

def run_ollama(prompt: str, model: str="llama3"):
    """
    If Ollama is available locally, run it.
    Otherwise return a simulated output so the pipeline does not crash.
    """
    if not ollama_available():
        # safe fallback for debugging retrieval/prompt wiring
        return json.dumps({
            "short_answer": "insufficient evidence",
            "primary_cause": "ollama not available",
            "failure_keywords": [],
            "signals": [],
            "confidence": 0.1
        })
    proc = subprocess.run(["ollama", "run", model], input=prompt, capture_output=True, text=True)
    return (proc.stdout or proc.stderr).strip()

SYSTEM_MSG = (
    "You are a maintenance engineering assistant. Answer ONLY from the provided context. "
    "If the context is insufficient, reply 'insufficient evidence'. "
    "Always cite manual title, section, and pages."
)

def stage1_prompt(datapoint: dict, lime_text: str, pred_label, prob, conf_score):
    """
    Stage 1: compact diagnosis + retrieval keywords (JSON-only)
    """
    return (
        SYSTEM_MSG + "\n\n"
        "You are an expert maintenance engineering assistant for rotating machinery.\n"
        "You MUST align your decision with the model prediction and explain why the machine is experiencing a FAILURE or NORMAL OPERATION using BOTH machine data and LIME explanation.\n\n"
        #"If the model predicts NORMAL and confidence is high (>0.8), you MUST NOT output failure.\n\n"
        f"LIME explanation:\n{lime_text}\n\n"
        "If not enough information, return short_answer='insufficient evidence'.\n\n"
        "STRICT OUTPUT: Return ONLY a single JSON object.\n"
        "Schema:\n"
        # "{\n"
        # '  "short_answer": "failure: <type>" | "normal operation" | "insufficient evidence",\n'
        # '  "primary_cause": "<one-line cause tied to parameters>",\n'
        # '  "failure_keywords": ["<kw1>", "<kw2>"],\n'
        # '  "signals": [{"name":"<feature>","value":"<val>","deviation":"high|low|normal"}],\n'
        # '  "confidence": <float 0.0-1.0>\n'
        # "}\n\n"
        "{\n"
        '  "short_answer": "...",\n'
        '  "primary_cause": "...",\n'
        '  "failure_keywords": [],\n'
        '  "signals": [],\n'
        '  "confidence": 0.0\n'
        "}\n\n"
        f"Machine data: {json.dumps(datapoint, indent=2)}\n"
        f"Model prediction: {pred_label} ({'Failure' if pred_label==1 else 'Normal'})\n"
        f"Model probability: {prob:.3f}\n"
        f"Model confidence: {conf_score:.3f}\n\n"

        
    )

def stage2_prompt(datapoint: dict, context_block: str, citations_meta: list, stage1_json: dict, lime_text):
    """
    Stage 2: grounded explanation + steps (JSON-only)
    """
    cites = "; ".join(
        [f"{m['manual']}, §{m.get('section') or 'General'}, pp.{m['pages']}" for m in citations_meta]
    )
    return (
        SYSTEM_MSG + "\n\n"
        "Explain the Stage 1 decision using ONLY the supplied Context Block.\n"
        f"LIME Explanation:\n{lime_text}\n\n"
        "If the context does not support the decision, say 'insufficient evidence' and recommend re-query.\n\n"
        "STRICT OUTPUT: Return ONLY a single JSON object.\n"
        "Schema:\n"
        "{\n"
        '  "answer": "<one short sentence status + reason>",\n'
        '  "steps": ["<action 1>", "<action 2>"],\n'
        '  "cautions": ["<safety note 1>"],\n'
        '  "citations": [{"manual":"<title>","section":"<section>","pages":"<start-end>","chunk_id":<int|null>}],\n'
        '  "evidence_summary": "<2-4 sentences grounded in context>",\n'
        '  "hallucination_check": {"unsupported_claims": [], "requery_suggested": <true|false>},\n'
        '  "confidence": <float 0.0-1.0>\n'
        "}\n\n"
        f"Machine data: {json.dumps(datapoint, indent=2)}\n"
        f"Stage1 JSON: {json.dumps(stage1_json, indent=2)}\n\n"
        "Context Block:\n"
        f"{context_block}\n\n"
        f"Required citations (from selected chunks): {cites}\n"
    )

## CELL 11 — Modify explain_one() to influence retrieval query + ranking

In [349]:
# <!-- Key logic:
# Compute prob_failure
# Compute lime_risk_score
# Determine SAFETY_MODE = True/False
# If safety mode:
# Use more aggressive retrieval: higher top_k, higher token_budget
# Add safety terms to query
# Re-rank chunks to prioritize safety content

# Else:
# Retrieve more normal-operation monitoring/inspection content -->

In [350]:
TOP_K_BASE = 8
TOKEN_BUDGET_BASE = 1200

def build_retrieval_query(datapoint, lime_triggers, safety_mode: bool):
    # numeric summary
    numeric_q = " ".join([
        f"torque={datapoint.get('torque', '')}",
        f"rpm={datapoint.get('rotational_speed', '')}",
        f"tool_wear={datapoint.get('tool_wear', '')}",
        f"temp_diff={datapoint.get('temp_diff', '')}",
        f"power_W={datapoint.get('power_W', '')}",
        f"overstrain={datapoint.get('overstrain_metric', '')}",
    ])

    # LIME feature hints
    lime_q = " ".join([f"feat:{t['feature']}" for t in lime_triggers])

    # Safety bias keywords
    if safety_mode:
        bias_q = " ".join(SAFETY_TERMS[:6] + FAILURE_TERMS[:6])
    else:
        bias_q = " ".join(["inspection", "monitoring", "normal operation", "preventive maintenance"])

    return f"{numeric_q} {lime_q} {bias_q}".strip()

def explain_one_safety_aware(index_in_test: int, model_for_ollama="llama3"):
    # scaled row for model + lime
    x_scaled = X_te_s.iloc[index_in_test]
    
    # --- LIME explanation (raw) ---
    lime_raw = lime_explain_one(x_scaled, num_features=8)

    # --- Convert to % contributions ---
    lime_percent = lime_weights_to_percent(lime_raw)

    # --- Natural language explanation ---
    lime_text = format_lime_explanation(lime_percent)
      
    x_full   = df.iloc[X_te_s.index[index_in_test]]

    prob = float(cal_rf.predict_proba(
        pd.DataFrame([x_scaled], columns=FEATURE_ORDER)
    )[:, 1][0])

    # Bayesian confidence estimation
    conf_score = bayesian_confidence(prob)
    conf_level = confidence_level(conf_score)

    # Safety-aware prediction label
    pred_label = int(prob >= tau_safety)  # 0 = normal, 1 = failure


    # LIME-risk score
    risk, triggers = lime_risk_score(x_scaled, lime_thresholds_scaled, features=top_feats, quantile_key="q70")

    # Safety decision gate (two-signal logic)
    CONF_THRESHOLD = 0.75  # trust model if confidence is high

    adjusted_risk = risk * (1 - conf_score)

    # safety_mode = (
    #     (prob >= tau_safety) or 
    #     ((risk >= MIN_HITS) and (conf_score < CONF_THRESHOLD))
    # )
    safety_mode = (
        (prob >= tau_safety) or 
        (adjusted_risk >= 1.0)
    )

    # Build datapoint for LLM prompt (raw-space is more interpretable)
    datapoint = {
        "air_temperature": float(x_full.get(COLS['air_temperature'], np.nan)),
        "process_temperature": float(x_full.get(COLS['process_temperature'], np.nan)),
        "temp_diff": float(x_full.get("temp_diff", np.nan)),
        "rotational_speed": float(x_full.get(COLS['rotational_speed'], np.nan)),
        "torque": float(x_full.get(COLS['torque'], np.nan)),
        "tool_wear": float(x_full.get(COLS['tool_wear'], np.nan)),
        "power_W": float(x_full.get("power_W", np.nan)),
        "overstrain_metric": float(x_full.get("overstrain_metric", np.nan))
    }

    # Retrieval settings influenced by safety mode
    top_k = TOP_K_BASE * (2 if safety_mode else 1)
    token_budget = TOKEN_BUDGET_BASE * (2 if safety_mode else 1)

    query = build_retrieval_query(datapoint, triggers, safety_mode=safety_mode)

    # Retrieve
    cand = hybrid_search(query, top_k=top_k)

    # Safety rerank
    if safety_mode:
        cand = safety_rerank(cand, boost_terms=SAFETY_TERMS + FAILURE_TERMS, boost_factor=0.80)

    context, meta = pack_context(cand, token_budget=token_budget)

    # Stage prompts (reuse your stage1/stage2 as-is)
    s1 = stage1_prompt(datapoint, lime_text, pred_label, prob, conf_score)
    s1_out = run_ollama(s1, model=model_for_ollama)

    try:
        s1_json = json.loads(s1_out) if s1_out.strip().startswith("{") else {"short_answer": s1_out, "failure_keywords": []}
    except Exception:
        s1_json = {"short_answer": s1_out, "failure_keywords": []}

    s2 = stage2_prompt(datapoint, context, meta, s1_json, lime_text)
    s2_out = run_ollama(s2, model=model_for_ollama)

    try:
        s2_json = json.loads(s2_out) if s2_out.strip().startswith("{") else {"answer": s2_out}
    except Exception:
        s2_json = {"answer": s2_out}

    return {
    "index_in_test": int(index_in_test),

    "lime_explanation": lime_percent,
    "lime_explanation_text": lime_text,

    # --- Prediction summary ---
    "pred_label": int(pred_label),   # 0 = normal, 1 = failure
    "prob_failure": float(prob),

    "bayesian_confidence_score": conf_score,
    "confidence_level": conf_level,
        
    "tau_safety": float(tau_safety),

    # --- Safety logic ---
    "lime_risk_score": int(risk),
    "lime_min_hits": int(MIN_HITS),
    "safety_mode": bool(safety_mode),

    # --- Explainability + Retrieval ---
    "lime_triggers": triggers,
    "retrieval_query": query,
    "citations_meta": meta,

    # --- LLM outputs ---
    "stage1": s1_json,
    "stage2": s2_json
}

## CELL 12 — Demo run

In [351]:
# ============================================================
# RETRIEVAL CORE (Jupyter-safe): conn + vector_search + bm25
# ============================================================

import numpy as np
import pandas as pd
import re
import joblib
from pathlib import Path

# --- Ensure DB connection exists ---
try:
    conn
except NameError:
    conn = sqlite3.connect(DB_PATH)

# --- Globals for BM25 (safe init) ---
try:
    bm25
except NameError:
    bm25 = None

try:
    bm25_doc_count
except NameError:
    bm25_doc_count = -1

def ensure_bm25(rows_df: pd.DataFrame, force: bool = False):
    """
    Build BM25 if missing or row count has changed.
    """
    global bm25, bm25_doc_count
    want = 0 if rows_df is None or rows_df.empty else len(rows_df)

    if force or bm25 is None or bm25_doc_count != want:
        try:
            from rank_bm25 import BM25Okapi
        except Exception as e:
            print("[warn] rank_bm25 not available:", e)
            bm25 = None
            bm25_doc_count = want
            return None

        texts = rows_df["text"].fillna("").tolist() if want else []
        tokenized = [t.split() for t in texts]
        bm25 = BM25Okapi(tokenized) if tokenized else None
        bm25_doc_count = want
    return bm25

def bm25_scores(rows_df: pd.DataFrame, query: str):
    """
    Returns dict {row_id: pseudo_distance} (smaller is better).
    """
    bm = ensure_bm25(rows_df)
    if bm is None or rows_df.empty:
        return {}

    ids = rows_df["id"].to_numpy(dtype=np.int64)
    scores = np.asarray(bm.get_scores(query.split()), dtype=float)

    n = min(len(ids), len(scores))
    if n == 0:
        return {}

    eps = 1e-9
    inv = 1.0 / (scores[:n] + eps)  # convert to distance-like
    return {int(ids[i]): float(inv[i]) for i in range(n)}

# --- Detect available vector backend based on on-disk artifacts ---
def detect_backend(index_dir: Path):
    faiss_ok = (index_dir / "faiss.index").exists()
    tfidf_ok = (index_dir / "tfidf_vec.joblib").exists() and (index_dir / "nn.joblib").exists()
    if faiss_ok:
        return "faiss"
    if tfidf_ok:
        return "tfidf"
    return "none"

def vector_search(query: str, top_k: int = 10):
    """
    Returns list of (chunk_id, distance). Lower distance = better.
    Uses:
      - FAISS if faiss.index exists
      - else TF-IDF+KNN if artifacts exist
      - else empty list
    """
    index_dir = INDEX
    backend = detect_backend(index_dir)

    # Must have row ids mapping
    row_ids_path = index_dir / "row_ids.npy"
    if not row_ids_path.exists():
        # rebuild row_ids.npy from DB
        ids_df = pd.read_sql_query("SELECT id FROM chunks", conn)
        np.save(row_ids_path, ids_df["id"].to_numpy(dtype=np.int64))

    row_ids = np.load(row_ids_path)
    if row_ids.size == 0:
        return []

    top_k = int(max(1, min(top_k, row_ids.size)))

    # --- FAISS path ---
    if backend == "faiss":
        try:
            import faiss
            index = faiss.read_index(str(index_dir / "faiss.index"))

            # Try SentenceTransformers local model first (if available in your notebook)
            qv = None
            try:
                from sentence_transformers import SentenceTransformer
                # IMPORTANT: use your local model path variable if you have it
                # If MODEL_DIR exists in your notebook, use it; else comment out.
                if "MODEL_DIR" in globals():
                    st = SentenceTransformer(str(MODEL_DIR))
                    qv = st.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
            except Exception:
                qv = None

            # Fallback: TF-IDF vector if exists
            if qv is None and (index_dir / "tfidf_vec.joblib").exists():
                vec = joblib.load(index_dir / "tfidf_vec.joblib")
                qv = vec.transform([query]).astype(np.float32).toarray()

            if qv is None:
                print("[warn] No embedding model available for FAISS query; returning empty.")
                return []

            D, I = index.search(qv, top_k)
            idxs = I[0]
            dists = D[0]
            out = []
            for j, local_i in enumerate(idxs):
                if local_i < 0:
                    continue
                out.append((int(row_ids[local_i]), float(dists[j])))
            return out

        except Exception as e:
            print("[warn] FAISS search failed, returning empty:", e)
            return []

    # --- TF-IDF KNN path ---
    if backend == "tfidf":
        try:
            vec = joblib.load(index_dir / "tfidf_vec.joblib")
            nn  = joblib.load(index_dir / "nn.joblib")
            qv  = vec.transform([query]).astype(np.float32)
            dist, ind = nn.kneighbors(qv, n_neighbors=top_k)
            ids_idx = row_ids[ind[0]]
            return [(int(rid), float(dist[0][k])) for k, rid in enumerate(ids_idx)]
        except Exception as e:
            print("[warn] TF-IDF search failed, returning empty:", e)
            return []

    # --- No index found ---
    return []

In [352]:
# ============================================================
# SAFETY: Ensure hybrid_search is defined (Jupyter-safe)
# ============================================================

def hybrid_search(query: str, top_k: int = 10):
    """
    Hybrid retrieval using:
    - Vector search (FAISS or TF-IDF)
    - BM25 keyword scores
    Returns a DataFrame with a 'hybrid_dist' column (lower = better)
    """
    rows_df = pd.read_sql_query("SELECT * FROM chunks", conn)
    if rows_df.empty:
        return rows_df.iloc[:0]

    # ---- Vector side ----
    vec_results = vector_search(query, top_k=top_k * 2)
    base = {rid: dist for rid, dist in vec_results}

    # ---- BM25 side ----
    bm_scores = bm25_scores(rows_df, query)
    for rid, bd in bm_scores.items():
        base[rid] = 0.5 * base.get(rid, bd) + 0.5 * bd

    if not base:
        return rows_df.iloc[:0]

    ranked = sorted(base.items(), key=lambda x: x[1])[:top_k]
    got = rows_df.set_index("id").loc[[rid for rid, _ in ranked]].reset_index()
    got["hybrid_dist"] = [dist for _, dist in ranked]
    return got


In [355]:
TEST_INDEX = 40
MODEL_NAME = "llama3"

res = explain_one_safety_aware(TEST_INDEX, model_for_ollama=MODEL_NAME)
print(json.dumps(res, indent=2)[:2500])

{
  "index_in_test": 40,
  "lime_explanation": [
    {
      "feature": "PWF",
      "weight": 0.5628327292947343,
      "percent": 25.081834793946882
    },
    {
      "feature": "OSF",
      "weight": 0.4929367158450148,
      "percent": 21.96701902213112
    },
    {
      "feature": "TWF",
      "weight": 0.4678768380295015,
      "percent": 20.85026144459505
    },
    {
      "feature": "HDF",
      "weight": 0.4343881343875707,
      "percent": 19.35788531134694
    },
    {
      "feature": "power_W",
      "weight": 0.08370119467392825,
      "percent": 3.7300239086986173
    }
  ],
  "lime_explanation_text": "This prediction is influenced 25% by Pwf, 22% by Osf, 21% by Twf, 19% by Hdf, 4% by Power W.",
  "pred_label": 1,
  "prob_failure": 1.0,
  "bayesian_confidence_score": 0.9773625583902263,
  "confidence_level": "Very High Confidence",
  "tau_safety": 0.64,
  "lime_risk_score": 1,
  "lime_min_hits": 2,
  "safety_mode": true,
  "lime_triggers": [
    {
      "feature": "te

In [354]:
label_meaning = {
    0: "Normal operation",
    1: "Failure operation"
}

print("\n=== DEMO SUMMARY ===")
print(f"Index in test set : {res['index_in_test']}")
print(f"Prediction label : {res['pred_label']} ({label_meaning[res['pred_label']]})")
print(f"Failure prob     : {res['prob_failure']:.3f}")
print(f"Safety threshold : {res['tau_safety']:.3f}")
print(f"Bayesian_Confidence_Score : {res['bayesian_confidence_score']:.3f}")
print(f"Confidence_level : {res['confidence_level']}")
print(f"LIME risk score  : {res['lime_risk_score']}")
print(f"Safety mode      : {res['safety_mode']}")


=== DEMO SUMMARY ===
Index in test set : 40
Prediction label : 1 (Failure operation)
Failure prob     : 1.000
Safety threshold : 0.640
Bayesian_Confidence_Score : 0.977
Confidence_level : Very High Confidence
LIME risk score  : 1
Safety mode      : True
